# 01 — Explore the Corner Maze

Interactive manual control of the corner-maze environment. Use this to build intuition for the maze layout, the action space, and what one trial cycle looks like.

## Setup (recommended: local VS Code)

1. Clone the repo and from its root run `pip install -e .` into your Python env (e.g. `ai-venv`).
2. Optional but recommended: run `scripts/setup_data.sh` once to copy the eye-image parquet from the legacy repo. Without it, the eye panels show placeholders; the maze still works.
3. Open this notebook in VS Code with the **Jupyter** extension and pick that env as the kernel.

## What you'll see

- **Maze** — top-down 13×13 view with the agent, barriers, cues, and wells.
- **Left / Right Eye** — first-person grayscale views from the agent's pose (only if the eye-image parquet is present).
- **Info panel** — current position, direction, step count, and cumulative reward.
- **Movement buttons** — click to move (Turn Left / Forward / Turn Right / Enter Well / Pause).
- **Keyboard (in VS Code / local Jupyter)** — click the maze to focus, then arrow keys move the agent. Space enters a well, `r` resets.

## Colab-in-browser is not officially supported

Colab's iframe-sandboxed widget framework + pinned ipywidgets 7.x cause unreliable rendering for this notebook (third-party widget consent prompts, blank cells, broken keyboard). Buttons may work after consenting; keyboard arrows will not. **For training scripts and other non-interactive components, Colab is fine** — it's only this manual-control UI that fights the iframe. Recommended cloud paths: GitHub Codespaces, or VS Code with a remote Jupyter kernel pointed at a GPU runtime.

In [ ]:
# Setup: assumes you ran `pip install -e .` from the repo root.
import numpy as np
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.env.constants import CORNER_POSES, EYE_IMG_SIZE
from minigrid.core.actions import Actions

print("Imports OK.")


## Pick a session type

The cell below uses `PI+VC f2 acquisition` by default. Other session types you can swap in:

**Exposure (open maze, no barriers):**
- `exposure` — Exposure A: open maze, all wells rewarded
- `exposure_b` — Exposure B: progressive barrier lowering

**PI+VC combined (fixed second turn, f2):**
- `PI+VC f2 acquisition` · `PI+VC f2 novel route` · `PI+VC f2 no cue` · `PI+VC f2 rotate` · `PI+VC f2 reversal`

**PI+VC combined (fixed first turn, f1):**
- `PI+VC f1 acquisition` · `PI+VC f1 novel route` · `PI+VC f1 no cue` · `PI+VC f1 rotate` · `PI+VC f1 reversal`

**PI-only (path integration, no visual cue):**
- `PI acquisition` · `PI novel route no cue` · `PI novel route cue` · `PI reversal no cue` · `PI reversal cue`

**VC-only (visual cue, rotating across arms):**
- `VC acquisition` · `VC novel route fixed` · `VC novel route rotate` · `VC reversal fixed` · `VC reversal rotate`


In [ ]:
#@title Interactive manual control
import io
import urllib.request
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage

# --- Configure the env here ---
SESSION_TYPE = "PI+VC f2 acquisition"
ORIENTATION = "N/NE"
GOAL = "NE"
SEED = 42

# --- Stage the eye-image parquet so the env can find it ---
# The 2 MB embeddings parquet isn't bundled with the pip wheel. We pull it
# from the legacy repo (public, raw URL) into ./data/dataframes/ on first
# run so the env's _load_embeddings() picks it up via its CWD-search path.
EYE_PARQUET = Path("data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet")
EYE_URL = ("https://github.com/ryangrg/corner-maze-rl-legacy/raw/main/"
           "data/dataframes/dual-indep-20260319-222411-embeddings-allposes.parquet")
if not EYE_PARQUET.is_file():
    EYE_PARQUET.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading eye-image embeddings (~2 MB) → {EYE_PARQUET}")
    urllib.request.urlretrieve(EYE_URL, EYE_PARQUET)
    print(f"  done ({EYE_PARQUET.stat().st_size // 1024} KB)")

env = CornerMazeEnv(
    render_mode="rgb_array",
    max_steps=10000,
    session_type=SESSION_TYPE,
    agent_cue_goal_orientation=ORIENTATION,
    start_goal_location=GOAL,
)
obs, info = env.reset(seed=SEED)

HAS_EYES = False
try:
    env._load_embeddings()
    HAS_EYES = True
except Exception as e:
    print(f"(eye images unavailable: {type(e).__name__}; maze panel still works)")

# --- Theme ---
BG_COLOR = "#333333"
TEXT_COLOR = "#ffffff"
MUTED_COLOR = "#aaaaaa"
DIR_NAMES = {0: "East", 1: "South", 2: "West", 3: "North"}
IMG_SIZE = 299  # 13 * 23 = 299, avoids grid-line aliasing

style_widget = widgets.HTML(f"""<style>
.maze-ui, .maze-ui * {{
    background-color: {BG_COLOR} !important;
    color: {TEXT_COLOR} !important;
    border-color: {BG_COLOR} !important;
}}
.maze-ui {{
    padding: 15px !important;
    border-radius: 8px !important;
    margin: 0 !important;
}}
.maze-ui .widget-image {{
    background-color: transparent !important;
}}
.maze-ui .panel-outline {{
    border: 1px solid #888 !important;
    border-color: #888 !important;
    box-sizing: border-box;
    padding: 4px;
}}
.maze-ui .section-divider {{
    height: 1px !important;
    background-color: #888 !important;
    margin: 12px 0 !important;
    width: 100% !important;
}}
/* Form inputs (dropdowns): outlined, slightly lighter than the BG so the
   white text still reads cleanly. Use compound selectors with .maze-ui so
   we beat the .maze-ui * blanket that forces border-color back to BG. */
.maze-ui .form-input select,
.maze-ui .form-input input {{
    background-color: #4a4a4a !important;
    color: #ffffff !important;
    border: 1px solid #888 !important;
    border-color: #888 !important;
    border-radius: 4px !important;
    padding: 2px 6px !important;
}}
/* Buttons: ipywidgets renders each Button as a single element with
   classes "jupyter-widgets jupyter-button widget-button" on the SAME div
   that .form-button is added to (not on a child). The previous selectors
   used descendant chains and didn't match anything. Target the element
   itself, with fallbacks for variant DOM structures. */
.maze-ui .form-button,
.maze-ui button.form-button,
.maze-ui .form-button > button,
.maze-ui .form-button > .jupyter-button {{
    background: #6a6a6a !important;
    background-image: none !important;
    background-color: #6a6a6a !important;
    color: #ffffff !important;
    border: 1px solid #d0d0d0 !important;
    border-color: #d0d0d0 !important;
    border-radius: 4px !important;
    box-shadow: inset 0 0 0 1px #d0d0d0 !important;
    font-weight: 500 !important;
    padding: 4px 12px !important;
    cursor: pointer !important;
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    line-height: 1 !important;
}}
.maze-ui .form-button:hover,
.maze-ui button.form-button:hover,
.maze-ui .form-button > button:hover,
.maze-ui .form-button > .jupyter-button:hover {{
    background: #7a7a7a !important;
    background-image: none !important;
    background-color: #7a7a7a !important;
}}
/* Strip backgrounds from inner elements (the FontAwesome <i> on icon
   buttons inherits .maze-ui * background-color, which paints a BG
   square behind the swirl). Force transparent so the button color
   shows through. */
.maze-ui .form-button *,
.maze-ui button.form-button *,
.maze-ui .form-button > button *,
.maze-ui .form-button > .jupyter-button * {{
    background: transparent !important;
    background-color: transparent !important;
    background-image: none !important;
    color: #ffffff !important;
}}
.maze-ui .form-field-label {{
    font-size: 12px !important;
    font-weight: 600 !important;
    color: #c8c8c8 !important;
    margin: 6px 0 2px 0 !important;
}}
.maze-ui .form-status {{
    font-size: 12px !important;
    color: #c8c8c8 !important;
    margin: 8px 0 0 0 !important;
    line-height: 1.5 !important;
}}
.maze-ui .form-status .session-name {{
    color: #ffffff !important;
    font-family: monospace, monospace !important;
}}
</style>""")

# --- State ---
cumulative_reward = 0.0
is_done = False
zero_eye = np.zeros((EYE_IMG_SIZE, EYE_IMG_SIZE), dtype=np.float64)


def frame_to_png_bytes(frame):
    img = PILImage.fromarray(frame).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def eye_to_png_bytes(eye_arr):
    """Render a [0,1] float eye image as PNG. Matches the env's pygame
    renderer: raw (arr * 255). Dim poses render dim — that's faithful to
    what the agent sees."""
    gray = (np.asarray(eye_arr, dtype=np.float64) * 255).astype(np.uint8)
    img = PILImage.fromarray(gray).resize((IMG_SIZE, IMG_SIZE), PILImage.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


# --- Widgets ---
maze_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
left_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
right_eye_widget = widgets.Image(format="png", width=IMG_SIZE, height=IMG_SIZE)
session_label = widgets.HTML(value="")
image_label = widgets.HTML(value="")
info_label = widgets.HTML(value="")


def update_display():
    global cumulative_reward, is_done

    frame = env.get_allocentric_frame(tile_size=23)
    maze_widget.value = frame_to_png_bytes(frame)

    if HAS_EYES:
        pose_label = env._get_pose_label()
        left_eye = env._pose_to_left_eye.get(pose_label, zero_eye)
        right_eye = env._pose_to_right_eye.get(pose_label, zero_eye)
    else:
        pose_label = "(eye images not loaded)"
        left_eye = zero_eye
        right_eye = zero_eye
    left_eye_widget.value = eye_to_png_bytes(left_eye)
    right_eye_widget.value = eye_to_png_bytes(right_eye)

    session_label.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<b>{env.session_type}</b></div>'
    )
    image_label.value = (
        f'<div style="font-size:13px; text-align:center; margin-top:4px;">'
        f'<code>{pose_label}</code></div>'
    )

    x, y = env.agent_pos
    d = env.agent_dir
    info_label.value = (
        f'<div style="font-size:14px; line-height:1.8;">'
        f'<b>Position:</b> ({x}, {y})<br>'
        f'<b>Direction:</b> {d} ({DIR_NAMES.get(d, "?")})<br>'
        f'<b>Step:</b> {env.step_count} / {env.max_steps}<br>'
        f'<b>Reward:</b> {cumulative_reward:.4f}<br>'
        f'<b>At corner:</b> {(x, y, d) in CORNER_POSES}<br>'
        f'<b>Done:</b> {is_done}'
        f'</div>'
    )


def on_action(action):
    global cumulative_reward, is_done
    if is_done:
        return
    obs, reward, terminated, truncated, info = env.step(action)
    cumulative_reward += reward
    is_done = terminated or truncated
    update_display()


def on_reset():
    global cumulative_reward, is_done
    cumulative_reward = 0.0
    is_done = False
    env.reset()
    update_display()


# (display label, env session_type) per training group.
SESSIONS_BY_GROUP = {
    "PI+VC f2": [
        ("Acquisition",  "PI+VC f2 acquisition"),
        ("Novel route",  "PI+VC f2 novel route"),
        ("No cue",       "PI+VC f2 no cue"),
        ("Rotate",       "PI+VC f2 rotate"),
        ("Reversal",     "PI+VC f2 reversal"),
    ],
    "PI+VC f1": [
        ("Acquisition",  "PI+VC f1 acquisition"),
        ("Novel route",  "PI+VC f1 novel route"),
        ("No cue",       "PI+VC f1 no cue"),
        ("Rotate",       "PI+VC f1 rotate"),
        ("Reversal",     "PI+VC f1 reversal"),
    ],
    "PI": [
        ("Acquisition",          "PI acquisition"),
        ("Novel route — no cue", "PI novel route no cue"),
        ("Novel route — cue",    "PI novel route cue"),
        ("Reversal — no cue",    "PI reversal no cue"),
        ("Reversal — cue",       "PI reversal cue"),
    ],
    "VC": [
        ("Acquisition",            "VC acquisition"),
        ("Novel route — fixed",    "VC novel route fixed"),
        ("Novel route — rotate",   "VC novel route rotate"),
        ("Reversal — fixed",       "VC reversal fixed"),
        ("Reversal — rotate",      "VC reversal rotate"),
    ],
}


def start_new_run(_button=None):
    """Rebuild the env from the run-config dropdowns and reset state."""
    global env, cumulative_reward, is_done
    new_session_type = session_dd.value
    try:
        env = CornerMazeEnv(
            render_mode="rgb_array",
            max_steps=10000,
            session_type=new_session_type,
            agent_cue_goal_orientation=ORIENTATION,
            start_goal_location=GOAL,
        )
        env.reset(seed=SEED)
        env._load_embeddings()
    except Exception as exc:
        info_label.value = (
            f'<div style="color:#f88; font-size:13px;">'
            f'<b>Could not start run:</b> {type(exc).__name__}: {exc}</div>'
        )
        return
    cumulative_reward = 0.0
    is_done = False
    update_display()


# --- Layout ---
# Mirror the legacy pygame render: three same-size square panels in a single
# row. Session label sits under the maze; pose label spans (and is centered
# under) just the two eye panels.
col_style = widgets.Layout(width=f"{IMG_SIZE}px", align_items="center")
maze_col = widgets.VBox(
    [widgets.HTML("<b>Maze</b>"), maze_widget, session_label],
    layout=col_style,
)
left_eye_col = widgets.VBox(
    [widgets.HTML("<b>Left Eye</b>"), left_eye_widget],
    layout=col_style,
)
right_eye_col = widgets.VBox(
    [widgets.HTML("<b>Right Eye</b>"), right_eye_widget],
    layout=col_style,
)
eye_pair = widgets.VBox(
    [widgets.HBox([left_eye_col, right_eye_col]), image_label],
    layout=widgets.Layout(align_items="center"),
)
images_row = widgets.HBox(
    [maze_col, eye_pair],
    layout=widgets.Layout(margin="0 0 6px 0"),
)

# --- Control panel: dropdowns + buttons (right column) ---
DEFAULT_GROUP = "PI+VC f2"
FIELD_W = "280px"

group_label = widgets.HTML('<div>Group</div>')
group_label.add_class("form-field-label")
group_dd = widgets.Dropdown(
    options=list(SESSIONS_BY_GROUP.keys()),
    value=DEFAULT_GROUP,
    layout=widgets.Layout(width=FIELD_W),
)
group_dd.add_class("form-input")

session_label_widget = widgets.HTML('<div>Session</div>')
session_label_widget.add_class("form-field-label")
session_dd = widgets.Dropdown(
    options=SESSIONS_BY_GROUP[DEFAULT_GROUP],
    value=SESSIONS_BY_GROUP[DEFAULT_GROUP][0][1],
    layout=widgets.Layout(width=FIELD_W),
)
session_dd.add_class("form-input")


def _on_group_change(change):
    new_group = change["new"]
    session_dd.options = SESSIONS_BY_GROUP[new_group]


group_dd.observe(_on_group_change, names="value")


def _format_status(session_type: str) -> str:
    return (
        '<div>Currently Running: '
        f'<span class="session-name">{session_type}</span></div>'
    )


status_label = widgets.HTML(value=_format_status(SESSION_TYPE))
status_label.add_class("form-status")


def start_new_run_with_status(_button=None):
    start_new_run(_button)
    status_label.value = _format_status(env.session_type)


# Option A button placement: primary right-aligned in its own row,
# Reset left-aligned on the row below as a de-emphasized secondary.
# Both share .form-button styling so they read as buttons (matching the
# segmented-control visual language) — the hierarchy comes from layout.
start_button = widgets.Button(description="Start New Run")
start_button.add_class("form-button")
start_button.layout = widgets.Layout(height="34px")
start_button.on_click(start_new_run_with_status)

reset_button = widgets.Button(
    description="Reset Current Run",
    icon="undo",
    tooltip="Reset Current Run (or press 'r')",
    layout=widgets.Layout(height="34px"),
)
reset_button.add_class("form-button")
reset_button.on_click(lambda _b: on_reset())




KEY_ACTION_MAP = {
    "ArrowUp": Actions.forward,
    "ArrowLeft": Actions.left,
    "ArrowRight": Actions.right,
    "ArrowDown": 4,    # ACTION_PAUSE
    " ": 3,            # ACTION_ENTER_WELL (space)
}


def on_key(event):
    key = event.get("key", "")
    if key == "r":
        on_reset()
        return
    action = KEY_ACTION_MAP.get(key)
    if action is not None:
        on_action(action)


# Keyboard handler (works in VS Code / local Jupyter; quietly no-ops
# in environments that don't load ipyevents). Click the maze to focus,
# then use arrow keys.
try:
    from ipyevents import Event
    if "_kb_event" in dir():
        try:
            _kb_event.close()
        except Exception:
            pass
    _kb_event = Event(source=images_row, watched_events=["keydown"], prevent_default_action=True)
    _kb_event.on_dom_event(on_key)
except Exception:
    pass  # buttons still work


# --- Direction / action buttons (primary control mechanism) ---
btn_left = widgets.Button(description="← Turn Left")
btn_left.add_class("form-button")
btn_left.layout = widgets.Layout(width="120px", height="34px")
btn_left.on_click(lambda _b: on_action(int(Actions.left)))

btn_forward = widgets.Button(description="↑ Forward")
btn_forward.add_class("form-button")
btn_forward.layout = widgets.Layout(width="120px", height="34px")
btn_forward.on_click(lambda _b: on_action(int(Actions.forward)))

btn_right = widgets.Button(description="→ Turn Right")
btn_right.add_class("form-button")
btn_right.layout = widgets.Layout(width="120px", height="34px")
btn_right.on_click(lambda _b: on_action(int(Actions.right)))

btn_well = widgets.Button(description="● Enter Well")
btn_well.add_class("form-button")
btn_well.layout = widgets.Layout(width="120px", height="34px")
btn_well.on_click(lambda _b: on_action(3))  # ACTION_ENTER_WELL

btn_pause = widgets.Button(description="▪ Pause")
btn_pause.add_class("form-button")
btn_pause.layout = widgets.Layout(width="120px", height="34px")
btn_pause.on_click(lambda _b: on_action(4))  # ACTION_PAUSE

movement_panel = widgets.HBox(
    [btn_left, btn_forward, btn_right, btn_well, btn_pause],
    layout=widgets.Layout(margin="0 0 8px 0", justify_content="flex-start"),
)

# --- Two-column info / control panel ---
left_info_col = widgets.VBox(
    [
        widgets.HTML(
            f'<div style="font-size:12px; color:{MUTED_COLOR}; margin-bottom:8px;">'
            f"<b>Controls:</b> Click the buttons below the maze to move the agent.<br>"
            f"In VS Code / local Jupyter you can also click the maze and use arrow keys, Space, r."
            f"</div>"
        ),
        info_label,
    ],
    layout=widgets.Layout(flex="1 1 0%", padding="10px"),
)
right_control_col = widgets.VBox(
    [
        widgets.HTML(
            '<div style="font-size:14px;"><b>Run Config</b></div>',
            layout=widgets.Layout(align_self="stretch"),
        ),
        group_label,
        group_dd,
        session_label_widget,
        session_dd,
        status_label,
        widgets.HBox(
            [start_button],
            layout=widgets.Layout(
                width=FIELD_W,
                justify_content="flex-start",
                margin="10px 0 6px 0",
            ),
        ),
        widgets.HBox(
            [reset_button],
            layout=widgets.Layout(
                width=FIELD_W,
                justify_content="flex-start",
            ),
        ),
    ],
    layout=widgets.Layout(
        flex="1 1 0%",
        padding="10px",
        align_items="flex-start",
    ),
)
info_panel = widgets.HBox(
    [left_info_col, right_control_col],
    layout=widgets.Layout(width="100%"),
)

divider = widgets.HTML(value="")
divider.add_class("section-divider")
ui = widgets.VBox([style_widget, images_row, divider, movement_panel, info_panel])
ui.add_class("maze-ui")
update_display()
display(ui)


## Try this

1. Run a complete trial: navigate from a start arm through one of the four corners and into the correct well (Space at the corner pose).
2. Watch how `Reward` changes — small negative per step (`-0.0005` forward, `-0.001` turn) and a `+1.061` spike on a correct well.
3. Try a wrong well and see that the trial doesn't end — the env keeps running until you visit the correct one.
4. Swap `SESSION_TYPE` for `"exposure"` and notice that all four wells reward (cycle-alternation).

## What's next

- **02_explore_yoked_data** — load the rodent behavioral dataset and plot trajectories.
- **03_compute_returns** — replay yoked sessions through the env to attach per-step reward and per-trial-with-ITI-start RTG.
- **04_train_dt** — train the Decision Transformer on `actions_with_returns.parquet`.
